#### **Customer Retention, Revenue Performance & Cancellation Analytics**

My objective is to identify revenue growth opportunities, valuable customer segments, and areas where cancellations may be affecting commercial performance.



These are the questions I plan to answer with the data.




1. How has revenue evolved over time: The objective is to Identify monthly revenue trends, growth patterns, and seasonality.
2. Growth patterns, How quickly is revenue changing?
3. Is grosss revenue growth driven by more customers, more orders, or higher average order values?	
4. What percentage of customers make repeat purchases, and how does this change over time?
5. How does repeat purchasing changes over time?
6. Which customer segments generate the most revenue? Identifying high-value customers through RMF segmentation.
7. Which products and countries generate the most revenue?
8. Which countries generate the most revenue?
9. How have cancellation rates evolved over time?
10. Which products experience the highest cancellation rates?
11. What is the financial impact of cancellations on reported revenue?
12. which customer segments contribute most to cancelled sales value?



7. How have cancellation rates evolved over time, and which products experience the highest cancellation rates?
8. What is the financial impact of cancellations on reported revenue, and which customer segments contribute most to cancelled sales value?



***Francis Ojinnaka***





**Dataset: UCI Online Retail II (2009–2011)** **more than 1 millon rows in teh dataset**


### **Key Commercial Takeaways**

**1. Revenue Performance & Growth Drivers**

Net revenue fluctuated significantly throughout 2010–2011, with recurring sales peaks in November (£1.40M in 2010 and £1.43M in 2011). Growth during peak periods was largely associated with increased customer activity and order volumes rather than consistently higher average order values.

**2. Customer Retention & Purchasing Behaviour**

Approximately **71.7% of identified customers made repeat purchases**. The monthly returning-customer share increased from 48% in January 2010 to 88% in November 2011, demonstrating the growing importance of customers with previous purchasing history.

**3. Customer Value & Revenue Concentration**

RFM segmentation revealed that **Champions and Big Spenders generated 49.2% of gross revenue despite representing only 4.3% of customers**. This highlights substantial revenue concentration among a small group of high-value customers.

**4. Product & Geographic Performance**

Regency Cakestand 3 Tier led merchandise revenue (£338.6K), while White Hanging Heart T-Light Holder demonstrated stronger unit demand.

The United Kingdom accounted for approximately **85.5% of gross revenue**, with Ireland and the Netherlands leading international contributions, highlighting significant geographic revenue concentration.

**5. Cancellation Performance & Financial Exposure**

The monthly cancellation invoice rate declined from 18.5% in January 2010 to 12.0% in November 2011.

Cancellation-related value represented approximately **2.9% of gross sales**. Big Spenders accounted for the largest share of cancelled value attributed to RFM segments (40.6%) and had the highest relative cancellation exposure (6.9%).

### Overall Takeaway

**The retailer's commercial performance is strongly influenced by repeat purchasing, a small group of high-value customers, and seasonal sales activity.** While cancellation incidence declined over the observation period, revenue concentration and cancellation exposure highlight opportunities to strengthen customer retention, protect high-value relationships, and improve commercial planning.


In [1]:
##Importing the libraries needed for the analysis

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 3, Finished, Available, Finished, False)

In [2]:
#Loading the dataseets from OneLake, appending the separate sheets and understanding the data

path = '/lakehouse/default/Files/Retail_data/online_retail_II.xlsx'

cols = ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID','Country']

sheets = pd.read_excel(
    path,
    sheet_name=None,  
    usecols=cols,
    parse_dates=['InvoiceDate'],
    dtype={'Invoice': 'str', 'StockCode': 'str', 'Price': 'float', 'Quantity': 'int32' , 'Customer ID': 'str'},
)



data = pd.concat(sheets.values(), ignore_index=True)


data = data.rename(columns={'Customer ID': 'Customer_ID'})
data.tail()

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 4, Finished, Available, Finished, False)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680,France
1067370,581587,POST,POSTAGE,1,2011-12-09 12:50:00,18.00,12680,France


In [3]:
## Data undertstanding and statistics
data.dtypes
data.info()
data.describe().round()

data['Country'].value_counts()
data.shape

data.columns 

#to understand the sum of NAs and what columns it appears
data.isna().sum()

# to know the number of unique vlaues in each columns
data.nunique()
data.describe().round()

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 5, Finished, Available, Finished, False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int32         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer_ID  824364 non-null   object        
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(1), int32(1), object(5)
memory usage: 61.1+ MB


,Quantity,InvoiceDate,Price
count,1067371.0,1067371,1067371.0
mean,10.0,2011-01-02 21:13:55.394028544,5.0
min,-80995.0,2009-12-01 07:45:00,-53594.0
25%,1.0,2010-07-09 09:46:00,1.0
50%,3.0,2010-12-07 15:28:00,2.0
75%,10.0,2011-07-22 10:23:00,4.0
max,80995.0,2011-12-09 12:50:00,38970.0
std,173.0,NaN,124.0


In [5]:
## data manipulation and feature engineering
## Removing 201112 data since it is not updated and coorect
## Removing the non-merchandise products since these are not real product description


data['StockCode']= data['StockCode'].str.title()
data['Description']= data['Description'].str.title()
data['Customer_ID']= data['Customer_ID'].str.strip()



data = data.assign(
   
    YearMonths = data['InvoiceDate'].dt.strftime('%Y%m'),
    Cancelled_incoice = np.where(data['Invoice'].str.contains('C'), 'Cancelled', 'Not Cancelled'),
    
    Revenue = data['Price'] * data['Quantity'],
    Non_Merchandise_Products = np.where(
        data['StockCode'].str.contains('C2'), 0, np.where(
        data['Description'].str.contains('Discount'), 0, np.where(
        data['Description'].str.contains('Bank Charges'), 0, np.where(
        data['Description'].str.contains('Adjust'), 0, np.where(
        data['Description'].str.contains('Amazon Fee'), 0, np.where(
        data['Description'].str.contains('Samples'), 0, np.where(
        data['StockCode'].str.contains('Post'), 0, np.where(
        data['Description'].str.contains('Manual'), 0, np.where(
        data['Description'].str.contains('Dotcom Postage'), 0,
        
        np.where(
        data['StockCode'].str.contains('Gift_00'), 0,1))))))))
)))
data.head()

data.query("Cancelled_incoice == 'Cancelled'").groupby(['Country'], as_index= False).agg(
    Revenue = ('Price', 'sum'),
    Counts = ('Price', 'size') ).sort_values(
        'Revenue', ascending = False).round(1)

#Filtering out the rows that I don't want in my analysis        

data = data.query(
    "YearMonths != '201112' and StockCode != 'TEST001' and StockCode != 'TEST002' and Non_Merchandise_Products == 1"
    )

data.head(10)




StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 11, Finished, Available, Finished, False)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer_ID,Country,YearMonths,Cancelled_incoice,Revenue,Non_Merchandise_Products
0,489434,85048,15Cm Christmas Glass Ball 20 Lights,12,2009-12-01 07:45:00,6.95,13085,United Kingdom,200912,Not Cancelled,83.4,1
1,489434,79323P,Pink Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,200912,Not Cancelled,81.0,1
2,489434,79323W,White Cherry Lights,12,2009-12-01 07:45:00,6.75,13085,United Kingdom,200912,Not Cancelled,81.0,1
3,489434,22041,"Record Frame 7"" Single Size",48,2009-12-01 07:45:00,2.10,13085,United Kingdom,200912,Not Cancelled,100.8,1
4,489434,21232,Strawberry Ceramic Trinket Box,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,200912,Not Cancelled,30.0,1
5,489434,22064,Pink Doughnut Trinket Pot,24,2009-12-01 07:45:00,1.65,13085,United Kingdom,200912,Not Cancelled,39.6,1
6,489434,21871,Save The Planet Mug,24,2009-12-01 07:45:00,1.25,13085,United Kingdom,200912,Not Cancelled,30.0,1
7,489434,21523,Fancy Font Home Sweet Home Doormat,10,2009-12-01 07:45:00,5.95,13085,United Kingdom,200912,Not Cancelled,59.5,1
8,489435,22350,Cat Bowl,12,2009-12-01 07:46:00,2.55,13085,United Kingdom,200912,Not Cancelled,30.6,1
9,489435,22349,"Dog Bowl , Chasing Ball Design",12,2009-12-01 07:46:00,3.75,13085,United Kingdom,200912,Not Cancelled,45.0,1


In [6]:
#Q1: How has net revenue evolved over time? (2009-2011)

revenue_trends = data.groupby(
    ['YearMonths'], as_index= False).agg(
        Net_Revenue = ('Revenue', 'sum')
    ).round(1).sort_values(['YearMonths', 'Net_Revenue'], ascending = [True, False])


fig = px.bar(revenue_trends,
 x="YearMonths",
 y='Net_Revenue', 
 labels=  True, 
 height = 700,
 text_auto='.3s',
  title="How has Net Revenue evolved over time?")
fig.show()

revenue_trends

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 13, Finished, Available, Finished, False)

,YearMonths,Net_Revenue
0,200912,781655.2
1,201001,606475.2
2,201002,527160.3
3,201003,754676.7
4,201004,638991.1
5,201005,608351.2
6,201006,672935.4
7,201007,617764.3
8,201008,663642.1
9,201009,844100.4


In [7]:
#Q2: Growth patterns, How quickly is revenue changing?

revenue_growth = data.groupby(
    ['YearMonths'], as_index= False).agg(
        Net_Revenue = ('Revenue', 'sum')
    ).round(1).sort_values(['YearMonths', 'Net_Revenue'], ascending = [True, False])

revenue_growth= revenue_growth.assign(
    Prev_Month_Revenue = revenue_growth['Net_Revenue'].shift(),
       Monthly_growth = lambda growth :(((growth['Net_Revenue']/growth['Prev_Month_Revenue'] -1)* 100)).round(1)
)

fig = px.bar(revenue_growth,
 x="YearMonths",
 y='Monthly_growth', 
 labels=  True, 
 height = 600,
 text_auto=True,
 color='Monthly_growth',

  title="Monthly Net Revenue Growth (%) — 2010–2011")
fig.show()




revenue_growth

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 14, Finished, Available, Finished, False)

,YearMonths,Net_Revenue,Prev_Month_Revenue,Monthly_growth
0,200912,781655.2,NaN,NaN
1,201001,606475.2,781655.2,-22.4
2,201002,527160.3,606475.2,-13.1
3,201003,754676.7,527160.3,43.2
4,201004,638991.1,754676.7,-15.3
5,201005,608351.2,638991.1,-4.8
6,201006,672935.4,608351.2,10.6
7,201007,617764.3,672935.4,-8.2
8,201008,663642.1,617764.3,7.4
9,201009,844100.4,663642.1,27.2


### Net Revenue Performance (2010–2011)

Net revenue fluctuated significantly throughout 2010–2011, with recurring growth momentum between September and November.

**November emerged as the strongest trading month** in both years, generating approximately £1.40M in 2010 and £1.43M in 2011, representing 2.1% year-over-year growth.

Monthly growth remained volatile, peaking at **51.6% in May 2011**, while January 2011 recorded the sharpest contraction (-50.1%).

**Commercial Takeaway:** Revenue performance suggests potential year-end seasonality despite substantial monthly fluctuations. These findings highlight opportunities to strengthen inventory planning, marketing activities, and operational capacity ahead of peak trading periods.

In [23]:
#Q3: Is grosss revenue growth driven by more customers, more orders, or higher average order values?	


Drivers = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['YearMonths'], as_index= False).agg(
        Gross_Revenue = ('Revenue', 'sum'),
        Active_Customers = ('Customer_ID', 'nunique'),
        Total_Orders =  ('Invoice', 'nunique')
    ).round(2).sort_values(['YearMonths', 'Gross_Revenue'], ascending = [True, False])

Drivers = Drivers.assign(

    Order_per_Customer = lambda orders:( orders['Total_Orders']/orders['Active_Customers']).round(2),
    Average_Order_Value = lambda AOV:( AOV['Gross_Revenue']/AOV['Total_Orders']).round(2)
)

Drivers



fig3 = go.Figure()

fig3.add_trace(
    go.Bar(
        x=Drivers['YearMonths'],
        y=Drivers['Average_Order_Value'],
        name = 'Average Order Value')
       )
    

fig3.add_trace(
    go.Bar(
        x=Drivers['YearMonths'],
        y= Drivers['Total_Orders'],
        name = 'Total Orders'
    ))

fig3.update_layout(
    title=dict(text='Monthly Average Orders and Total Orders', x=0.5),  
    xaxis_title='YearMonth',
    yaxis_title='Share',
    legend_title_text='Metric'
)

fig3.show()


fig1 = px.bar(Drivers,
 x="YearMonths",
 y='Active_Customers', 
 labels=  True, 
 height = 500,
 text_auto=True,
  title="Monthly active Customers")
fig1.show()





Drivers



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 58, Finished, Available, Finished, False)

,YearMonths,Gross_Revenue,Active_Customers,Total_Orders,Order_per_Customer,Average_Order_Value
0,200912,801216.47,952,1689,1.77,474.37
1,201001,613940.81,702,1059,1.51,579.74
2,201002,539744.69,773,1207,1.56,447.18
3,201003,764333.85,1051,1675,1.59,456.32
4,201004,649016.32,939,1451,1.55,447.29
5,201005,645645.39,965,1520,1.58,424.77
6,201006,699316.02,1034,1651,1.60,423.57
7,201007,634896.99,924,1563,1.69,406.20
8,201008,676060.72,910,1425,1.57,474.43
9,201009,871878.34,1136,1819,1.60,479.32



**Key Findings**

- **Volume-Driven Growth:** Revenue growth was closely associated with increases in active customers and order volumes, particularly during peak trading months.
- **Strong November Performance:** In November 2011, gross revenue increased by 32.9%, supported by a 22.0% increase in active customers and a 12.8% rise in orders per customer.
- **AOV Fluctuations:** Average order value varied throughout the period and declined by 3.3% in November 2011, despite strong revenue growth.

**Commercial Insight**

Revenue growth during peak periods appears primarily driven by increased customer participation and purchasing frequency rather than consistently higher spending per order.

**Executive Takeaway**

Customer activity and order frequency were key drivers of peak-period revenue growth, highlighting opportunities to strengthen customer acquisition and encourage repeat purchasing.

In [9]:
#Q4: What percentage of customers make repeat purchases, and how does this change over time?

customers  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['Customer_ID'], as_index= False).agg(
        Gross_Revenue = ('Revenue', 'sum'),
    
        Total_Orders =  ('Invoice', 'nunique')
    ).round(2).sort_values(['Total_Orders'], ascending = [ False])

customers= customers.assign(


    Average_Order_Value = lambda AOV:( AOV['Gross_Revenue']/AOV['Total_Orders']).round(2),
    Customer_type = lambda rpt: np.where(
        rpt['Total_Orders'] > 1, 'Repeat', 'One Time'
    )
)

customers.head(20)



repeaters = customers.query("Customer_type == 'Repeat'").groupby(['Customer_type'])['Customer_ID'].nunique()
repeaters = repeaters.sum()

total_customers = customers.groupby(['Customer_type'])['Customer_ID'].nunique()
total_customers = total_customers.sum()


Repeat_rate = (repeaters/total_customers * 100).round(1)
display(Repeat_rate)



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 16, Finished, Available, Finished, False)

71.7


**Key Finding**

Approximately 71.7% of identified customers made more than one purchase during the observation period, while 28.3% purchased only once.

**Commercial Insight**

The high proportion of repeat purchasers suggests that returning customers play an important role in the retailer's purchasing activity.

However, repeat purchasing does not necessarily indicate strong customer loyalty, as the dataset does not capture customer satisfaction, competitor purchases, or behavior beyond the observation period.

**Executive Takeaway**

Nearly three-quarters of observed customers made repeat purchases, highlighting the importance of returning customers to the retailer's commercial activity.

In [10]:
#Q5. how does repeat purchasing changes over time?

#first task is to get the min year purchase for each customers

min_year = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['Customer_ID'], as_index= False).agg(
       
             First_purchase = ('YearMonths', 'min')
    
       
    )


#second task is to calculate know the # of repeater purchase by Customer ID and Year month
repeat_purchase = data.query(
    "Cancelled_incoice == 'Not Cancelled' ").groupby(
    ['Customer_ID','YearMonths'], as_index= False).agg(
        Gross_Revenue = ('Revenue', 'sum'),    
        Total_Orders =  ('Invoice', 'nunique')
    ).round(2).sort_values(['YearMonths'], ascending = [ True])


#next task is to join the 2 tables together
repeat_purchaser = repeat_purchase.merge(
    min_year,
    on = ['Customer_ID'],
    how = 'inner'
) 

#last task is to create the conditional column


repeat_purchaser= repeat_purchaser.assign(


   Next_purhase =lambda rpt: np.where(
        rpt['YearMonths'] ==   rpt['First_purchase'], 'New', 'Returning'
   
))

#prelim Result
repeat_purchaser.head(20)

#now grouping just by yearmonth cos i wnat the trend over time

active_customer_only = repeat_purchaser.groupby(['YearMonths'],as_index = False).agg(
    Active_Customer = ('Customer_ID', 'nunique')
)

active_customer_only



returning_customer_only = repeat_purchaser.query(
    "Next_purhase == 'Returning'").groupby(['YearMonths'],as_index = False).agg(
    Returning_customers = ('Customer_ID', 'nunique')
)

returning_customer_only



## Merging with returning.
Customer_rate = active_customer_only.merge(
    returning_customer_only,
    how = 'inner',
    on = ['YearMonths']
)


Customer_rate['Returning_Customer_Rate'] =( Customer_rate['Returning_customers']/Customer_rate['Active_Customer'] * 100).round()

Customer_rate

fig1 = px.line(Customer_rate,
 x="YearMonths",
 y='Returning_Customer_Rate', 
 labels=  True, 
 height = 500,
  markers=True,
 


  title="Returning Customer Trends (2010–2011)")
fig1.show()


Customer_rate

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 18, Finished, Available, Finished, False)

,YearMonths,Active_Customer,Returning_customers,Returning_Customer_Rate
0,201001,702,334,48.0
1,201002,773,396,51.0
2,201003,1051,611,58.0
3,201004,939,645,69.0
4,201005,965,710,74.0
5,201006,1034,767,74.0
6,201007,924,739,80.0
7,201008,910,747,82.0
8,201009,1136,897,79.0
9,201010,1492,1117,75.0



**Key Findings**

- **Increasing Returning-Customer Share:** The monthly returning-customer rate increased from **48% in January 2010 to 88% in November 2011**, a rise of 40 percentage points.
- **Strong Repeat Purchasing Activity:** Returning customers accounted for approximately 82–91% of monthly active customers throughout 2011.
- **Growing Returning-Customer Base:** The number of returning customers increased from 334 in January 2010 to 1,469 in November 2011.

**Commercial Insight**

Returning customers represented an increasingly large proportion of monthly purchasing activity, indicating growing reliance on customers with previous purchase history.

This highlights the commercial importance of maintaining relationships with existing customers while continuing to attract new buyers.

**Executive Takeaway**

Returning customers became the dominant share of monthly active buyers, reaching 88% by November 2011, highlighting the importance of repeat purchasing to the retailer's commercial activity.


In [11]:
#Q6: Which customer segments generate the most revenue? Identifying high-value customers through RMF segmentation.



dt_RFM  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['Customer_ID'], as_index= False).agg(
        Gross_Monetary= ('Revenue', 'sum'),
    
        Frequency =  ('Invoice', 'nunique'),
        Max_Purchase_Date = ('InvoiceDate', 'max')
    ).round(2).sort_values(['Frequency'], ascending = [ False])



## to perform an RFM analysis, I need a refrence date to clauclat the datediff between the invoicedate and reference date
## I am going to use the max Invoicedate and add an additional day as my refernce
## Add score to my RFM backet where 5 is the highest and 1 is the lowest


dt_RFM= dt_RFM.assign(


  reference_date =data['InvoiceDate'].max()   + pd.to_timedelta(1, unit = 'D'), #Using the maximum date in my table + 1
  Recency_Days = lambda recency : ( recency['reference_date'] -  recency['Max_Purchase_Date']).dt.days,
  Recency_Score = lambda score : np.where(  score['Recency_Days'] >= 50, 1, 
  np.where(  score['Recency_Days'] >= 30, 2,
  np.where(  score['Recency_Days'] >= 20, 3,
  np.where(  score['Recency_Days'] > 5, 4,
  np.where(  score['Recency_Days'] <= 5, 5, 0
  ))))),

  Frequency_score = lambda frq : np.where(frq['Frequency']>= 50, 5, 
  np.where(frq['Frequency']>= 30, 4,
  np.where(frq['Frequency']>= 20, 3,
    np.where(frq['Frequency']> 5, 2,
      np.where(frq['Frequency']<= 5, 1,0

      ))))),
      Monetary_score = lambda mon: np.where(
        mon['Gross_Monetary'] >= 40000, 5,
        np.where(
        mon['Gross_Monetary'] >= 10000, 4,
             np.where(
        mon['Gross_Monetary'] >= 5000, 3,
      np.where(
        mon['Gross_Monetary'] >= 1500, 2,
        np.where(
        mon['Gross_Monetary'] < 1500, 1,0



      ))))

  ))
  
## grouping customers based on theri RFM score to customer segments brackets  
   

dt_RFM

Conditions = [
    (dt_RFM['Monetary_score'].isin([5,4]) ) & (dt_RFM['Frequency_score'].isin([5,4])) & (dt_RFM['Recency_Score'].isin([5,4])),
    (dt_RFM['Frequency_score'].isin([5,4])) & (dt_RFM['Recency_Score'].isin([5,4]) ),
    (dt_RFM['Monetary_score'].isin([5,4])),
    (dt_RFM['Monetary_score'].isin([3,2])) & (dt_RFM['Frequency_score'] == 2) & (dt_RFM['Recency_Score'].isin([5,4]) ),
    (dt_RFM['Monetary_score'].isin([5,4,3])) & (dt_RFM['Frequency_score'].isin([5,4,3])) & (dt_RFM['Recency_Score'].isin([3,2,1]) ),
    (dt_RFM['Monetary_score'].isin([1,2])) & (dt_RFM['Frequency_score'].isin([1,2])),
    (dt_RFM['Monetary_score'].isin([3,2])),
 
]


segments = ['Champions', 'Loyal Customers', 'Big Spenders','Potential Loyalists' , 'At-Risk Customers', 'Low-Value Customers', 'Medium Spenders']



dt_RFM['Customer_segments'] = np.select(Conditions, segments, default= 'None')

dt_RFM.query("Customer_segments == 'None'")

dt_RFM['Customer_segments'].value_counts()

dt_RFM.reset_index()



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 20, Finished, Available, Finished, False)

,index,Customer_ID,Gross_Monetary,Frequency,Max_Purchase_Date,reference_date,Recency_Days,Recency_Score,Frequency_score,Monetary_score,Customer_segments
0,2508,14911,269825.69,364,2011-11-30 15:08:00,2011-12-01 17:42:00,1,5,5,5,Champions
1,386,12748,51124.84,313,2011-11-29 18:14:00,2011-12-01 17:42:00,1,5,5,5,Champions
2,5387,17841,68307.46,207,2011-11-29 17:52:00,2011-12-01 17:42:00,1,5,5,5,Champions
3,2903,15311,115300.09,204,2011-11-29 13:12:00,2011-12-01 17:42:00,2,5,5,5,Champions
4,722,13089,112056.24,199,2011-11-28 14:53:00,2011-12-01 17:42:00,3,5,5,5,Champions
...,...,...,...,...,...,...,...,...,...,...,...
5822,412,12774,83.70,1,2010-06-22 10:38:00,2011-12-01 17:42:00,527,1,1,1,Low-Value Customers
5823,4485,16922,611.08,1,2010-05-16 11:24:00,2011-12-01 17:42:00,564,1,1,1,Low-Value Customers
5824,4483,16920,355.45,1,2010-10-24 14:07:00,2011-12-01 17:42:00,403,1,1,1,Low-Value Customers
5825,2809,15217,426.24,1,2010-05-25 13:51:00,2011-12-01 17:42:00,555,1,1,1,Low-Value Customers


In [12]:

# Final output: on answering the customer segments that generated the most revenue? 

segments = dt_RFM.groupby(
    ['Customer_segments'], as_index= False).agg(

    Gross_Revenue = ('Gross_Monetary', 'sum'),
    Active_Customers = ('Customer_ID', 'nunique')
).sort_values('Gross_Revenue', ascending = False)


segments = segments.assign(
    Share_of_Revenue = lambda tl : (tl['Gross_Revenue'] /(tl['Gross_Revenue'].sum())* 100).round(1),
    Share_of_CustomerBase = lambda Customer : (Customer['Active_Customers'] /(Customer['Active_Customers'].sum())* 100).round(1),
    Revenue_per_Customer =  (segments['Gross_Revenue']/segments['Active_Customers']).round(1)
    )

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=segments['Customer_segments'],
        y=segments['Share_of_CustomerBase'],
        name = '# Customer Share')
       )
    

fig.add_trace(
    go.Bar(
        x=segments['Customer_segments'],
        y= segments['Share_of_Revenue'],
        name = 'Revenue Share'
    ))

fig.update_layout(
    title=dict(text='Gross Revenue Contribution by RFM Customer Segment', x=0.5),  
    xaxis_title='Customer Segments',
    yaxis_title='Share (%)',
    legend_title_text='Metric'
)

fig.show()

segments



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 21, Finished, Available, Finished, False)

,Customer_segments,Gross_Revenue,Active_Customers,Share_of_Revenue,Share_of_CustomerBase,Revenue_per_Customer
2,Champions,5068881.78,95,30.0,1.6,53356.7
3,Low-Value Customers,4888596.67,4864,28.9,83.5,1005.1
1,Big Spenders,3247760.86,155,19.2,2.7,20953.3
5,Medium Spenders,1754459.60,266,10.4,4.6,6595.7
6,Potential Loyalists,1530279.85,391,9.0,6.7,3913.8
0,At-Risk Customers,273950.86,37,1.6,0.6,7404.1
4,Loyal Customers,157530.68,19,0.9,0.3,8291.1



**Key Findings**

- **Revenue Concentration:** Champions and Big Spenders collectively generated **49.2% of gross revenue**, despite representing only 4.3% of the customer base.
- **High-Value Customers:** Champions contributed the largest revenue share (29.9%) from just 95 customers, demonstrating substantial revenue concentration.
- **Low-Value Segment:** Low-Value Customers represented 83.5% of customers but contributed only 28.9% of revenue.

**Commercial Insights**

The findings highlight significant revenue dependence on a relatively small group of high-value customers, making their continued purchasing activity commercially important.

The large Low-Value segment also presents opportunities to investigate ways to increase purchase frequency and customer spending.

**Executive Takeaway**

Nearly half of gross revenue is generated by just 4.3% of customers, highlighting the importance of retaining high-value customers while exploring growth opportunities across the wider customer base.


In [13]:
#Q7: Which products  generate the most gross revenue?

dt_prod  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['Description'], as_index= False).agg(
        Gross_Revenue= ('Revenue', 'sum'),
        Total_Quantity= ('Quantity', 'sum'),
         Total_Orders =  ('Invoice', 'nunique'),
        Active_Csutomers =  ('Customer_ID', 'nunique'),
    ).round(2).sort_values(['Gross_Revenue'], ascending = [ False])



dt_prod = dt_prod.assign(
    Revenue_Share = lambda pct :round(( pct['Gross_Revenue']/pct['Gross_Revenue'].sum()* 100), 2),
    Qty_Share = lambda pct :round(( pct['Total_Quantity']/pct['Total_Quantity'].sum()* 100), 2)
)

## The top 10 products with the most generated revenue
dt_prod_top = dt_prod.nlargest(10,'Gross_Revenue')

dt_prod_top


fig2 = go.Figure()

fig2.add_trace(
    go.Bar(
        x=dt_prod_top['Description'],
        y=dt_prod_top['Qty_Share'],
        name = 'Quantity share')
       )
    

fig2.add_trace(
    go.Bar(
        x=dt_prod_top['Description'],
        y= dt_prod_top['Revenue_Share'],
        name = 'Revenue Share'
    ))

fig2.update_layout(
    title=dict(text='Top 10 Merchandise Products — Revenue Share vs. Quantity Share (2009–2011)', x=0.5),  
    xaxis_title='Products',
    yaxis_title='Share (%)',
    legend_title_text='Metric',
    
    xaxis_tickangle=-45,  
)

fig2.show()

dt_prod_top

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 23, Finished, Available, Finished, False)

,Description,Gross_Revenue,Total_Quantity,Total_Orders,Active_Csutomers,Revenue_Share,Qty_Share
3958,Regency Cakestand 3 Tier,338556.38,27144,3861,1307,1.74,0.25
5356,White Hanging Heart T-Light Holder,264716.53,95910,5399,1487,1.36,0.88
2538,Jumbo Bag Red Retrospot,148708.66,78180,3220,854,0.76,0.72
3336,Party Bunting,148268.57,28259,2648,892,0.76,0.26
327,Assorted Colour Bird Ornament,130025.18,80487,2761,1003,0.67,0.74
3304,Paper Chain Kit 50'S Christmas,116270.83,34492,1902,878,0.60,0.32
1058,Chilli Lights,82850.17,16385,1109,301,0.42,0.15
2856,Medium Ceramic Top Storage Jar,81448.21,77826,233,135,0.42,0.72
4107,Rotating Silver Angels T-Light Hldr,71748.14,31632,708,288,0.37,0.29
2543,Jumbo Bag Strawberry,68988.32,37036,1840,546,0.35,0.34




**Key Findings**

- **Revenue Leadership:** Regency Cakestand 3 Tier generated the highest gross revenue (338.6K), accounting for 1.74% of merchandise revenue, followed by White Hanging Heart T-Light Holder (264.7K; 1.36%).
- **Volume vs. Revenue:** White Hanging Heart recorded significantly higher sales volume (95.9K units) than Regency Cakestand (27.1K units), despite generating less revenue.
- **Revenue Distribution:** No individual product among the top 10 contributed more than 2% of total merchandise revenue, suggesting limited dependence on any single product.

**Commercial Insight**

Product revenue performance is influenced by both selling price and sales volume. The results highlight the importance of evaluating revenue contribution alongside unit demand when assessing product performance.

**Executive Takeaway**

Regency Cakestand leads merchandise revenue generation, while White Hanging Heart demonstrates stronger unit demand. Revenue is distributed across multiple products rather than concentrated in a single best-selling item.


In [14]:
#Q8: Which countries generate the most revenue?

dt_country  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['Country'], as_index= False).agg(
        Gross_Revenue= ('Revenue', 'sum'),
        Total_Quantity= ('Quantity', 'sum'),
         Total_Orders =  ('Invoice', 'nunique'),
        Active_Customers =  ('Customer_ID', 'nunique'),
    ).round(2).sort_values(['Gross_Revenue'], ascending = [ False])



dt_country = dt_country.assign(
    Revenue_Share = lambda pct :round(( pct['Gross_Revenue']/pct['Gross_Revenue'].sum()* 100), 2),
    Qty_Share = lambda pct :round(( pct['Total_Quantity']/pct['Total_Quantity'].sum()* 100), 2),
    Revenue_per_customer = (dt_country['Gross_Revenue']/dt_country['Active_Customers']).round(0)
)

dt_country = dt_country.nlargest(10,'Gross_Revenue' )



fig6 = go.Figure()

fig6.add_trace(
    go.Scatter(
        x=dt_country['Country'],
        y=dt_country['Qty_Share'],
        name = 'Quantity share')
       )
    

fig6.add_trace(
    go.Bar(
        x=dt_country['Country'],
        y= dt_country['Revenue_Share'],
        name = 'Revenue Share'
    ))

fig6.update_layout(
    title=dict(text='Top 10 Countries — Revenue Share vs. Quantity Share (2009–2011) including the UK', x=0.5),  
    xaxis_title='Country',
    yaxis_title='Share (%)',
    legend_title_text='Metric'
)

fig6.show()

dt_country

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 25, Finished, Available, Finished, False)

,Country,Gross_Revenue,Total_Quantity,Total_Orders,Active_Customers,Revenue_Share,Qty_Share,Revenue_per_customer
40,United Kingdom,16661844.63,8845714,36368,5316,85.47,81.52,3134.0
11,EIRE,621787.74,336202,571,3,3.19,3.10,207263.0
26,Netherlands,538224.64,376404,214,22,2.76,3.47,24465.0
15,Germany,381805.58,221933,736,106,1.96,2.05,3602.0
14,France,310409.88,270194,581,92,1.59,2.49,3374.0
0,Australia,168484.66,104392,89,15,0.86,0.96,11232.0
34,Spain,98324.58,50223,142,37,0.50,0.46,2657.0
36,Switzerland,94270.29,52733,85,22,0.48,0.49,4285.0
35,Sweden,86115.04,88514,98,19,0.44,0.82,4532.0
10,Denmark,68409.29,237680,41,11,0.35,2.19,6219.0


### Geographic Revenue Performance (2009–2011)

**Key Findings**

- **UK Market Dominance:** The United Kingdom generated approximately **16.66 million**, accounting for 85.5% of total gross revenue and 91.8% of units sold.
- **International Revenue:** Ireland (EIRE) and the Netherlands were the largest international markets, contributing 3.2% and 2.8% of gross revenue, respectively.
- **Revenue vs. Volume:** The UK's revenue share was lower than its quantity share, suggesting lower average revenue per unit compared with the international sales mix.
- **Customer Concentration:** Ireland and the Netherlands generated substantial revenue from relatively few identified customers, indicating potentially large purchasing volumes per customer.

**Commercial Insights**

The retailer's strong dependence on the UK market creates geographic revenue concentration. International markets contribute smaller revenue shares.

**Executive Takeaway**

The UK dominates revenue generation, accounting for 85.5% of gross sales. Ireland and the Netherlands lead international contributions, highlighting both geographic revenue concentration and potential opportunities to evaluate overseas customer demand.


In [15]:
#Q8: Which  International Markets generate the most revenue ?

dt_country  = data.query(
    "Cancelled_incoice == 'Not Cancelled' and Country != 'United Kingdom' ").groupby(
    ['Country'], as_index= False).agg(
        Gross_Revenue= ('Revenue', 'sum'),
        Total_Quantity= ('Quantity', 'sum'),
         Total_Orders =  ('Invoice', 'nunique'),
        Active_Customers =  ('Customer_ID', 'nunique'),
    ).round(2).sort_values(['Gross_Revenue'], ascending = [ False])



dt_country = dt_country.assign(
    Revenue_Share = lambda pct :round(( pct['Gross_Revenue']/pct['Gross_Revenue'].sum()* 100), 2),
    Qty_Share = lambda pct :round(( pct['Total_Quantity']/pct['Total_Quantity'].sum()* 100), 2),
    Revenue_per_customer = (dt_country['Gross_Revenue']/dt_country['Active_Customers']).round(0)
)

dt_country = dt_country.nlargest(10,'Gross_Revenue' )



fig6 = go.Figure()

fig6.add_trace(
    go.Scatter(
        x=dt_country['Country'],
        y=dt_country['Qty_Share'],
        name = 'Quantity share')
       )
    

fig6.add_trace(
    go.Bar(
        x=dt_country['Country'],
        y= dt_country['Revenue_Share'],
        name = 'Revenue Share'
    ))

fig6.update_layout(
    title=dict(text='Top 10 International Markets — Revenue Share vs. Quantity Share (2009–2011)', x=0.5),  
    xaxis_title='Country',
    yaxis_title='Share (%)',
    legend_title_text='Metric'
    
    
)

fig6.show()

dt_country

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 27, Finished, Available, Finished, False)

,Country,Gross_Revenue,Total_Quantity,Total_Orders,Active_Customers,Revenue_Share,Qty_Share,Revenue_per_customer
11,EIRE,621787.74,336202,571,3,21.95,16.76,207263.0
26,Netherlands,538224.64,376404,214,22,19.00,18.77,24465.0
15,Germany,381805.58,221933,736,106,13.48,11.06,3602.0
14,France,310409.88,270194,581,92,10.96,13.47,3374.0
0,Australia,168484.66,104392,89,15,5.95,5.20,11232.0
34,Spain,98324.58,50223,142,37,3.47,2.50,2657.0
36,Switzerland,94270.29,52733,85,22,3.33,2.63,4285.0
35,Sweden,86115.04,88514,98,19,3.04,4.41,4532.0
10,Denmark,68409.29,237680,41,11,2.41,11.85,6219.0
3,Belgium,56034.04,33796,139,29,1.98,1.68,1932.0


**Key Findings**

- **Leading International Markets:** Ireland (EIRE) and the Netherlands generated the highest international gross revenue, contributing **21.95% and 19.00%**, respectively, of the top 10 international markets shown.
- **Revenue vs. Volume:** Germany accounted for 11.06% of units sold but only 13.48% of revenue, suggesting a lower average revenue per unit.
- **High-Value Markets:** The Netherlands generated 19.00% of revenue from just 18.77% of units sold, indicating a comparatively high revenue contribution per unit.
- **Revenue Concentration:** Ireland and the Netherlands collectively contributed approximately **41%** of gross revenue across the top 10 international markets.

**Commercial Insight**

International markets demonstrate different purchasing patterns, with some generating revenue primarily through sales volume and others through higher revenue per unit.

These differences highlight opportunities to investigate product mix, customer purchasing behavior, and geographic revenue concentration.

**Executive Takeaway**

Ireland and the Netherlands lead international revenue generation, while Germany demonstrates stronger unit demand relative to revenue. This highlights distinct commercial dynamics across overseas markets.


In [21]:
#Q9: How have cancellation rates evolved over time, and which products experience the highest cancellation rates?



dt_canc  = data.query(
    "Cancelled_incoice == 'Cancelled'").groupby(
    ['YearMonths'], as_index= False).agg(
        cancelled_orders =  ('Invoice', 'nunique')
    ).round(2).sort_values(['YearMonths'], ascending = [True])


dt_canc


all_invoices  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['YearMonths'], as_index= False).agg(
    Sales_orders =  ('Invoice', 'nunique')
    ).round(2).sort_values(['YearMonths'], ascending = [True])


all_invoices = all_invoices.merge(
   dt_canc,
   on = ['YearMonths'],
   how = 'left'

)


all_invoices['Total_Invoices'] = all_invoices['cancelled_orders']+ all_invoices['Sales_orders']
all_invoices['Cancellation Invoice Rate (%)'] = (all_invoices['cancelled_orders']/all_invoices['Total_Invoices'] * 100).round(1)





fig = px.bar(all_invoices,
 x="YearMonths",
 y='Cancellation Invoice Rate (%)', 
 labels=  True, 
 height = 700,
 text_auto='.3s',
  title="Monthly Cancellation Invoice Rate (%)")


fig.show()


all_invoices


StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 39, Finished, Available, Finished, False)

,YearMonths,Sales_orders,cancelled_orders,Total_Invoices,Cancellation Invoice Rate (%)
0,200912,1689,379,2068,18.3
1,201001,1059,241,1300,18.5
2,201002,1207,219,1426,15.4
3,201003,1675,351,2026,17.3
4,201004,1451,263,1714,15.3
5,201005,1520,387,1907,20.3
6,201006,1651,319,1970,16.2
7,201007,1563,314,1877,16.7
8,201008,1425,242,1667,14.5
9,201009,1819,319,2138,14.9




**Key Findings**

- **Declining Cancellation Rates:** Cancellation incidence decreased from 18.5% in January 2010 to 12.0% in November 2011, a reduction of **6.5 percentage points**.
- **Peak Cancellation Activity:** May 2010 recorded the highest cancellation rate (20.3%), followed by January 2011 (18.4%).
- **Recent Improvement:** Cancellation rates declined from 15.9% in June 2011 to 12.0% in November 2011, despite some monthly fluctuations.

**Commercial Insight**

The overall decline suggests an improving cancellation profile relative to sales activity. However, cancellation levels remained notable, highlighting opportunities to investigate products and customer segments associated with frequent transaction reversals.

**Executive Takeaway**

Cancellation incidence generally improved over the period, declining by 6.5 percentage points between January 2010 and November 2011, although fluctuations persisted.

In [17]:
#Q10: Which products experience the highest cancellation rates?

dt_canc_product  = data.query(
    "Cancelled_incoice == 'Cancelled'").groupby(
    ['StockCode'], as_index= False).agg(
        Cancelled_Qty =  ('Quantity', 'sum')
    ).round(2).sort_values(['Cancelled_Qty'], ascending = [True])


dt_canc_sold  = data.query(
    "Cancelled_incoice == 'Not Cancelled'").groupby(
    ['StockCode'], as_index= False).agg(
        Total_Qty_sold =  ('Quantity', 'sum')
    ).round(2).sort_values(['Total_Qty_sold'], ascending = [False])


dt_canc_product['Cancelled_Qty'] = abs(dt_canc_product['Cancelled_Qty'])
dt_canc_product.head()



StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 31, Finished, Available, Finished, False)

,StockCode,Cancelled_Qty
1769,23166,74494
2327,84347,18750
269,21088,7140
273,21096,7008
27,16047,5184


In [18]:
total_prodcuts = dt_canc_sold.merge(
    dt_canc_product[['StockCode', 'Cancelled_Qty']],
    on = ['StockCode'],
    how = 'inner'
)
#joining with the description to have a descriptive test

desc_table= data[['StockCode', 'Description']].drop_duplicates()

desc_table


total_prodcuts['Cancelled_rate'] =( total_prodcuts['Cancelled_Qty']/total_prodcuts['Total_Qty_sold']* 100).round(1)


total_prodcuts = total_prodcuts.sort_values('Cancelled_rate')
total_prodcuts = total_prodcuts.nsmallest(10, 'Cancelled_rate')

total_prodcuts

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 33, Finished, Available, Finished, False)

,StockCode,Total_Qty_sold,Cancelled_Qty,Cancelled_rate
2822,23113,-295,946,-320.7
2820,79341,-180,448,-248.9
2842,79323W,-1826,1088,-59.6
2814,21450,-18,10,-55.6
2824,79323G,-343,187,-54.5
2823,79323S,-340,161,-47.4
2834,79323Lp,-1185,458,-38.6
2828,85036C,-666,193,-29.0
2815,45016,-29,6,-20.7
2839,79323B,-1428,248,-17.4


In [19]:
#Q11. What is the financial impact of cancellations on reported revenue?

dt_Cancelled_value =data.query(
    "Cancelled_incoice == 'Cancelled'")['Revenue'].abs().sum()

Gross_sales_value = data.query(
    "Cancelled_incoice != 'Cancelled'")['Revenue'].sum()


cancellations_value_rate = (dt_Cancelled_value/Gross_sales_value * 100).round(1)

cancellations_value_rate





StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 34, Finished, Available, Finished, False)

2.9



**Key Finding**

Cancellation transactions represented approximately **2.9% of gross reported sales revenue** during the analysis period.

**Commercial Insight**

Although cancellation-related value represents a relatively small proportion of gross sales, it remains an important area for monitoring financial adjustments and potential revenue leakage.

**Executive Takeaway**

Cancellation adjustments were equivalent to 2.8% of gross sales, highlighting the importance of monitoring cancellations and identifying the products and customer segments most affected.

In [20]:

#Q12: which customer segments contribute most to cancelled sales value?

##Calcualting the canceleled value per customer
dt_cancelled = (
    data.query("Cancelled_incoice == 'Cancelled'")
    .groupby("Customer_ID", as_index=False)
    .agg(Cancelled_Value=("Revenue", "sum"))
).round()

dt_cancelled["Cancelled_Value"] = (
    dt_cancelled["Cancelled_Value"].abs()

)

dt_cancelled


# Merge it to the dt_HFM

cancelled_cus = dt_RFM.merge(
    dt_cancelled,
    how = 'left',
    on = ['Customer_ID']
)

cancelled_cus['Cancelled_Value'] = cancelled_cus['Cancelled_Value'].fillna(0)

cancelled_cus

## calcualte by customer_segments 
seg = cancelled_cus.groupby(
    ['Customer_segments'], as_index = False
).agg(
    Gross_Revenue=("Gross_Monetary", "sum"),
        Cancelled_Value=("Cancelled_Value", "sum"),
        Customers=("Customer_ID", "nunique")
    )

seg = seg.assign(
    Cancellation_Contribution = lambda canc :( (canc['Cancelled_Value']/canc['Cancelled_Value'].sum()) * 100). round(2),
    Cancellation_value_ratio = lambda cvr :( (cvr['Cancelled_Value']/cvr['Gross_Revenue']) * 100). round(2),
   
)


fig8 = go.Figure()

fig8.add_trace(
    go.Bar(
        x=seg['Customer_segments'],
        y=seg['Cancellation_Contribution'],
        name = 'Cancellation_Contribution')
       )
    

fig8.add_trace(
    go.Bar(
        x=seg['Customer_segments'],
        y= seg['Cancellation_value_ratio'],
        name = 'Cancellation_value_ratio'
    ))

fig8.update_layout(
    title=dict(text='Cancellation Value Contribution vs. Revenue Exposure by RFM Customer Segment', x=0.5),  
    xaxis_title='Customer_segments',
    yaxis_title='Share (%)',
    legend_title_text='Metric'
)

fig8.show()


seg

StatementMeta(, 41a75f7c-6068-4c41-b28b-faa6c4b351c6, 35, Finished, Available, Finished, False)

,Customer_segments,Gross_Revenue,Cancelled_Value,Customers,Cancellation_Contribution,Cancellation_value_ratio
0,At-Risk Customers,273950.86,5241.0,37,0.95,1.91
1,Big Spenders,3247760.86,223643.0,155,40.59,6.89
2,Champions,5068881.78,138364.0,95,25.11,2.73
3,Low-Value Customers,4888596.67,110554.0,4864,20.06,2.26
4,Loyal Customers,157530.68,2640.0,19,0.48,1.68
5,Medium Spenders,1754459.60,41386.0,266,7.51,2.36
6,Potential Loyalists,1530279.85,29215.0,391,5.30,1.91


**Key Findings**

- **Cancellation Concentration:** Big Spenders accounted for the largest share of cancelled sales value (40.6%), followed by Champions (25.1%) and Low-Value Customers (20.1%).
- **High-Value Customer Exposure:** Big Spenders and Champions collectively contributed **65.7% of attributed cancellation value**.
- **Relative Cancellation Exposure:** Big Spenders recorded the highest cancellation-to-revenue ratio (6.9%), compared with 2.7% for Champions and 2.3% for Low-Value Customers.

**Commercial Insight**

Cancellation-related value is concentrated among high-value customers, particularly Big Spenders. Their relatively high cancellation exposure highlights the importance of investigating transaction reversals within this segment.

**Executive Takeaway**

Big Spenders represent both the largest contributor to cancelled sales value and the highest relative cancellation exposure, making them a priority for further investigation.